# 📗 German Credit Dataset — Complete Study Notes

**A beginner-friendly walkthrough of the Statlog German Credit dataset**

Welcome! This notebook is your complete study guide for the **German Credit dataset** — the second of the two datasets in our FairML project (the first, COMPAS, has its own companion notebook with the identical structure, so you can study them side by side).

By the end of this notebook you will understand:

1. **What the German Credit dataset is** and what decision it models
2. **What every column means**, in plain language (including how we decode the cryptic original file)
3. **The shape and structure** of the data (rows, columns, types, missing values, duplicates)
4. **Exploratory Data Analysis (EDA)** — distributions, class imbalance, correlations
5. **Fairness concerns** — the protected attribute (**sex**) and the biases baked into this dataset
6. **The exact preprocessing workflow** our project (`German.py`) uses before training models

💡 **How to use this notebook:** Run each code cell in order (`Shift + Enter`). Every code cell is preceded by *what we're about to do* and followed by *what the output means*.


---
## 1. Dataset Overview

### What is the German Credit dataset?

The **Statlog German Credit** dataset was collected in Germany in the 1970s–80s by Professor Hans Hofmann. It contains **1,000 loan applications** at a German bank. For each applicant we know their financial situation (checking account, savings, credit amount requested…), personal circumstances (age, sex, housing, job…), and — the key part — whether the bank's experience with that customer turned out **good** (they repaid) or **bad** (they were a credit risk).

It is one of the classic benchmark datasets of machine learning, hosted at the **UCI Machine Learning Repository**, and a standard benchmark in fairness research because it contains personal attributes (sex, age) that anti-discrimination laws protect.

### What problem does it help solve?

The prediction task: **given an applicant's information, predict whether giving them a loan is a good or a bad credit risk.** Banks face this decision thousands of times a day — this is called **credit scoring**.

It is a **binary classification** problem, just like COMPAS. And just like COMPAS, our FairML project doesn't only want accurate predictions — it wants predictions that don't systematically disadvantage protected groups (here: by **sex**).

### What does each row represent?

**One row = one loan applicant** at the German bank, with their personal and financial details plus the actual outcome of their loan.

### What is the target variable?

Our project creates a target column called **`risk`** (from the original file's good/bad label):

| Value | Meaning |
|-------|---------|
| `1` | **Bad** credit risk — the loan turned out badly (the *adverse* outcome) |
| `0` | **Good** credit risk — the customer repaid reliably |

> 🔁 **Deliberate parallel to COMPAS:** in both datasets `1` = the adverse event (re-offending / bad credit). Keeping the encoding consistent lets us compare fairness results across the two studies directly.


---
## 2. Setup — Importing Our Tools

**What we're about to do:** Import the same toolbox as in the COMPAS notebook — **pandas** (tables), **numpy** (math), **matplotlib/seaborn** (charts) — and fix a **colorblind-safe color palette** used consistently in every chart.


In [ ]:
# Import the libraries (our data-science toolbox)
import pandas as pd              # tables of data
import numpy as np               # numerical arrays and math
import matplotlib.pyplot as plt  # basic plotting
import seaborn as sns            # prettier statistical plots

import warnings
warnings.filterwarnings("ignore")  # hide non-critical warnings

# --- Fixed, colorblind-safe palette used for EVERY chart in this notebook ---
COLORS = sns.color_palette("colorblind")
C_GOOD = COLORS[0]   # blue   -> good credit (label 0)
C_BAD  = COLORS[1]   # orange -> bad credit  (label 1)
C_MAIN = COLORS[0]   # blue   -> single-series charts

sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams["figure.dpi"] = 100

print("Libraries loaded successfully!")


**What the output means:** All libraries are pre-installed in Google Colab, so this should just print the success message.


---
## 3. Loading the Data (and Decoding It)

### Why this dataset needs a decoding step

The original UCI file (`german.data`) is **cryptic**: categorical values are stored as codes like `A11`, `A43`, `A93` that mean nothing without the documentation. For example `A11` means *"checking account balance < 0 DM"* (DM = Deutsche Mark, the pre-euro German currency).

Our project uses a **cleaned, human-readable version** (`data/german_credit_data.csv`, the popular Kaggle version) with 9 friendly columns — but that cleaned file has **no target column**, so `German.py` pulls the good/bad label from the original UCI file (the same 1,000 rows in the same order).

**In this notebook we do both steps ourselves**, so it runs anywhere (including Colab) with no local files:
1. Download the raw file straight from the UCI repository.
2. Decode the `A`-codes into the same readable columns our project uses.

This is also great practice — real-world data almost always arrives in inconvenient formats.

**What we're about to do:** Download `german.data` from UCI. It has no header row and is separated by spaces, so we supply the 21 column names ourselves (20 attributes + the target), exactly as `German.py` does.


In [ ]:
# Column names for the raw UCI file (20 attributes + target),
# matching the names used in our project's German.py
statlog_cols = [
    "existing_checking", "duration", "credit_history", "purpose", "credit_amount",
    "savings", "employment", "installment_rate", "personal_status_sex", "other_debtors",
    "residence_since", "property", "age", "other_installment", "housing",
    "existing_credits", "job", "num_dependents", "telephone", "foreign_worker", "target"
]

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data"
statlog = pd.read_csv(url, sep=" ", header=None, names=statlog_cols)

print("Shape:", statlog.shape)
statlog.head()


**What the output means:** **1,000 rows × 21 columns**, but full of codes like `A11`, `A34`, `A43` — unreadable without a legend. The last column `target` uses `1` = good credit and `2` = bad credit (UCI's convention).

**What we're about to do next:** Decode the codes into plain English, producing exactly the 9 readable columns our project's cleaned CSV has (`Age`, `Sex`, `Job`, `Housing`, `Saving accounts`, `Checking account`, `Credit amount`, `Duration`, `Purpose`) plus the `risk` target. The decoding maps come from the official UCI documentation.


In [ ]:
# --- Decoding maps from the official UCI documentation ---

checking_map = {                      # balance of the checking (current) account
    "A11": "little",                  # ... < 0 DM (overdrawn!)
    "A12": "moderate",                # 0 to 200 DM
    "A13": "rich",                    # >= 200 DM (or salary paid in for >= 1 year)
    "A14": np.nan,                    # no checking account -> unknown
}
savings_map = {                       # balance of the savings account
    "A61": "little",                  # < 100 DM
    "A62": "moderate",                # 100 to 500 DM
    "A63": "quite rich",              # 500 to 1000 DM
    "A64": "rich",                    # >= 1000 DM
    "A65": np.nan,                    # unknown / no savings account
}
sex_map = {                           # original column mixes sex & marital status;
    "A91": "male",                    # male, divorced/separated
    "A92": "female",                  # female, divorced/separated/married
    "A93": "male",                    # male, single
    "A94": "male",                    # male, married/widowed
    "A95": "female",                  # female, single
}                                     # -> we keep only the sex part
housing_map = {"A151": "rent", "A152": "own", "A153": "free"}
job_map = {                           # job skill level, as an ordered number
    "A171": 0,                        # unskilled, non-resident
    "A172": 1,                        # unskilled, resident
    "A173": 2,                        # skilled employee / official
    "A174": 3,                        # highly qualified / management
}
purpose_map = {                       # what the loan is for
    "A40": "car", "A41": "car",       # new car / used car -> merged to "car"
    "A42": "furniture/equipment", "A43": "radio/TV",
    "A44": "domestic appliances", "A45": "repairs",
    "A46": "education", "A48": "vacation/others",
    "A49": "business", "A410": "vacation/others",
}

# --- Build the readable table (same columns as data/german_credit_data.csv) ---
data = pd.DataFrame({
    "Age":              statlog["age"],
    "Sex":              statlog["personal_status_sex"].map(sex_map),
    "Job":              statlog["job"].map(job_map),
    "Housing":          statlog["housing"].map(housing_map),
    "Saving accounts":  statlog["savings"].map(savings_map),
    "Checking account": statlog["existing_checking"].map(checking_map),
    "Credit amount":    statlog["credit_amount"],
    "Duration":         statlog["duration"],
    "Purpose":          statlog["purpose"].map(purpose_map),
})

# --- Target: UCI codes good=1, bad=2. We encode 1 = BAD credit risk
#     (the adverse outcome), mirroring COMPAS where 1 = re-offended. ---
data["risk"] = (statlog["target"] == 2).astype(int)

print("Decoded table shape:", data.shape)
data.head(10)


**What the output means:** Now the table is human-readable: ages, `male`/`female`, `own`/`rent`, loan purposes in words, real amounts. Notice some `NaN` values in `Saving accounts` and `Checking account` — those are the applicants with *no known account* (codes `A65`/`A14`). We'll deal with them in Section 5.

> 📝 **Note:** our project's `data/german_credit_data.csv` is this exact table (it's the well-known Kaggle cleaned version). We rebuilt it from the source so the notebook is self-contained and you've seen *where every value comes from*.


---
## 4. Column-by-Column Explanation

Our working table has **9 features + 1 target**:

| Column | Type | What it means | Why it might matter for credit risk |
|--------|------|---------------|--------------------------------------|
| `Age` | Numerical (integer) | Applicant's age in years (19–75) | Younger applicants have shorter credit histories and less stable income → historically higher risk. ⚠️ Also a protected attribute! |
| `Sex` | Categorical | `male` or `female` (derived from the original combined "personal status & sex" code) | Should **not** matter for creditworthiness — that's exactly why it's our ⭐ **protected attribute** for fairness analysis |
| `Job` | Categorical, but stored as an ordered number 0–3 | Skill level: 0 = unskilled non-resident, 1 = unskilled resident, 2 = skilled, 3 = highly qualified/management | Higher skill → more stable income → lower risk |
| `Housing` | Categorical | `own` (owns home), `rent`, or `free` (lives for free, e.g. with family) | Homeowners tend to be financially settled; also serves as informal collateral |
| `Saving accounts` | Categorical (ordered) | Savings balance: `little` (<100 DM), `moderate` (100–500), `quite rich` (500–1000), `rich` (>1000), or missing = unknown/none | Savings are a safety cushion — more savings, safer loan |
| `Checking account` | Categorical (ordered) | Checking balance: `little` (<0 DM — overdrawn!), `moderate` (0–200), `rich` (≥200), or missing = no account | Day-to-day financial health; an overdrawn account is a warning sign. One of the strongest predictors in this dataset |
| `Credit amount` | Numerical (integer) | Loan amount requested, in Deutsche Mark (250–18,424 DM) | Bigger loans are harder to repay → higher risk |
| `Duration` | Numerical (integer) | Loan repayment period in **months** (4–72) | Longer loans mean more time for things to go wrong → higher risk |
| `Purpose` | Categorical | What the loan is for: car, radio/TV, furniture/equipment, business, education, repairs, domestic appliances, vacation/others | Consumption loans vs. investment loans can carry different risk |
| **`risk`** | **Target** (0/1) | `1` = bad credit risk, `0` = good | **This is what we predict** |

> ⚠️ **Careful with `Job`:** it *looks* numeric (0–3) and pandas treats it as a number, so our pipeline **scales it like a number** rather than one-hot encoding it. That's defensible because the levels are genuinely ordered (more skill = higher number), but it's worth knowing it's really an encoded category.

> 📖 **Columns we left behind:** the raw UCI file has 11 more attributes (credit history, employment length, installment rate, other debtors, property, other installment plans, existing credits, dependents, telephone, foreign worker). The cleaned Kaggle version — and therefore our project — uses only the 9 above. Fewer features = a simpler, more interpretable fairness study, at some cost in accuracy.


**What we're about to do:** Look at the actual values of each categorical column, with counts — the fastest way to get familiar with a dataset.


In [ ]:
# Value counts for every categorical feature
for col in ["Sex", "Job", "Housing", "Saving accounts", "Checking account", "Purpose"]:
    print(f"--- {col} ---")
    print(data[col].value_counts(dropna=False), "\n")  # dropna=False also shows missing


**What the output means:**
- **Sex:** 690 male vs. 310 female — men are overrepresented 2:1 (remember this for the fairness section).
- **Job:** most applicants (630) are level 2, "skilled employee".
- **Housing:** most people (713) own their home.
- **Saving accounts:** `little` dominates (603); 183 are `NaN` (unknown/no account).
- **Checking account:** 394 are `NaN` (no checking account) — the single most-missing column.
- **Purpose:** cars (337), radio/TV (280) and furniture (181) dominate — mostly consumer loans.


---
## 5. Data Shape and Structure

**What we're about to do:** The same four health checks as in the COMPAS notebook: size, data types, missing values, duplicates.


In [ ]:
# 1) Size
print(f"Rows (loan applicants): {data.shape[0]}")
print(f"Columns:                {data.shape[1]}")

# 2) Data types
print("\nData types:")
print(data.dtypes)


**What the output means:** **1,000 rows × 10 columns** — tiny by modern standards (COMPAS has 7× more rows). Small datasets are quick to train on but give **noisier statistics**: results can shift noticeably with a different random split.

Types: `Age`, `Job`, `Credit amount`, `Duration`, `risk` are integers; `Sex`, `Housing`, `Saving accounts`, `Checking account`, `Purpose` are text (`object`).


In [ ]:
# 3) Missing values per column
missing = data.isnull().sum()
print(missing[missing > 0].to_string() if missing.any() else "No missing values")
print(f"\nShare of rows missing 'Checking account': {data['Checking account'].isna().mean()*100:.1f}%")
print(f"Share of rows missing 'Saving accounts':  {data['Saving accounts'].isna().mean()*100:.1f}%")


**What the output means:** Exactly two columns have missing values — `Saving accounts` (183, ~18%) and `Checking account` (394, ~39%). But this missingness is **informative, not accidental**: it means *"this applicant has no such account (or the bank doesn't know)"*. Having no checking account is itself a meaningful fact about someone's finances!

**How our project handles it** (`German.py`): `data.fillna("unknown")` — missing becomes its own category `"unknown"`, which the one-hot encoder then treats like any other value. This is the standard trick for informative missingness in categorical columns. Let's apply it now.


In [ ]:
# Treat "missing" as its own category, exactly as German.py does
data = data.fillna("unknown")

print("Missing values after fillna:", data.isnull().sum().sum())
print("\n'Checking account' categories now:")
print(data["Checking account"].value_counts())


**What the output means:** Zero missing values remain, and `Checking account` now has four honest categories: `unknown` (394), `little` (274), `moderate` (269), `rich` (63). No information was thrown away.


In [ ]:
# 4) Duplicates
print("Exact duplicate rows:", data.duplicated().sum())


**What the output means:** `0` duplicates — all 1,000 applications are unique. (With only 10 columns, exact duplicates would have been possible by coincidence, so this was worth checking.)


---
## 6. Exploratory Data Analysis (EDA)

Same plan as the COMPAS notebook:

6.1 Summary statistics
6.2 Target distribution (class imbalance)
6.3 Distributions of the numeric features
6.4 Categorical features vs. the target
6.5 Correlation analysis
6.6 Interesting patterns & observations


### 6.1 Summary Statistics

**What we're about to do:** `.describe()` on the numeric columns — count, mean, spread, min/max, quartiles.


In [ ]:
# Summary statistics of numeric columns
data.describe().round(2)


**What the output means:**
- **`Age`**: 19–75, mean ≈ 35.5, median 33 → skews young, like COMPAS.
- **`Job`**: mean ≈ 1.9 → the typical applicant is "skilled" (level 2).
- **`Credit amount`**: median 2,320 DM but max 18,424 DM → strongly **right-skewed**; a few very large loans.
- **`Duration`**: 4–72 months, median 18 → mostly short consumer loans.
- **`risk`**: mean 0.30 → **30% of loans went bad**. That's our class imbalance, examined next.


### 6.2 Target Variable Distribution (Class Imbalance Analysis)

**What we're about to do:** Count and plot good vs. bad credit risks.


In [ ]:
# Class counts and percentages
counts = data["risk"].value_counts().sort_index()
pcts = (counts / len(data) * 100).round(1)

print(f"  0 = good credit: {counts[0]:>4} applicants ({pcts[0]}%)")
print(f"  1 = bad credit:  {counts[1]:>4} applicants ({pcts[1]}%)")

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(["Good credit (0)", "Bad credit (1)"], counts.values,
              color=[C_GOOD, C_BAD], width=0.6)
for bar, pct in zip(bars, pcts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 12,
            f"{bar.get_height():,.0f}  ({pct}%)", ha="center", fontsize=11)
ax.set_ylabel("Number of applicants")
ax.set_title("Target: credit risk")
ax.set_ylim(0, counts.max() * 1.18)
sns.despine()
plt.tight_layout()
plt.show()


**What the output means:** **700 good vs. 300 bad — a 70/30 imbalance**, clearly stronger than COMPAS's 55/45. Two practical consequences:

1. **The lazy baseline is high:** always predicting "good" scores 70% accuracy. So when our models report ~70–75% accuracy, always compare against this 70% floor.
2. **Errors on the minority class hide easily:** a model can look accurate while missing most bad risks. That's why the project also tracks fairness/error metrics, not just accuracy.

> 📖 **Fun fact from the official documentation:** the dataset comes with a cost matrix stating that approving a *bad* customer is **5× more costly** than rejecting a *good* one. Our project doesn't use the cost matrix, but it's a reminder that in credit scoring, the two error types are not equally bad.


### 6.3 Distributions of the Numeric Features

**What we're about to do:** Histograms of `Age`, `Credit amount`, and `Duration`.


In [ ]:
# Histograms of the three key numeric features
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(data["Age"], bins=28, color=C_MAIN, edgecolor="white")
axes[0].set_title("Age")
axes[0].set_xlabel("Years")
axes[0].set_ylabel("Number of applicants")

axes[1].hist(data["Credit amount"], bins=30, color=C_MAIN, edgecolor="white")
axes[1].set_title("Credit amount")
axes[1].set_xlabel("Loan size (Deutsche Mark)")

axes[2].hist(data["Duration"], bins=24, color=C_MAIN, edgecolor="white")
axes[2].set_title("Duration")
axes[2].set_xlabel("Repayment period (months)")

sns.despine()
plt.tight_layout()
plt.show()


**What the output means:**
- **Age**: right-skewed, bulk in the 20s–40s.
- **Credit amount**: heavily right-skewed — most loans are small (< 4,000 DM), few are huge. `StandardScaler` will compress that tail but not remove the skew.
- **Duration**: spikes at round numbers (12, 24, 36, 48 months) — loans are sold in standard term lengths. Recognizing such "human fingerprints" in data is a useful EDA skill.


**What we're about to do next:** Split those distributions by the target to see which features separate good from bad risks.


In [ ]:
# Feature distributions split by outcome
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for value, color, label in [(0, C_GOOD, "Good credit"), (1, C_BAD, "Bad credit")]:
    subset = data[data["risk"] == value]
    axes[0].hist(subset["Age"], bins=28, alpha=0.6, color=color,
                 label=label, edgecolor="white")
    axes[1].hist(subset["Credit amount"], bins=30, alpha=0.6, color=color,
                 label=label, edgecolor="white")
    axes[2].hist(subset["Duration"], bins=24, alpha=0.6, color=color,
                 label=label, edgecolor="white")

for ax, title, xlab in zip(axes,
        ["Age by outcome", "Credit amount by outcome", "Duration by outcome"],
        ["Years", "Deutsche Mark", "Months"]):
    ax.set_title(title); ax.set_xlabel(xlab); ax.legend()
axes[0].set_ylabel("Number of applicants")

sns.despine()
plt.tight_layout()
plt.show()

# Numeric confirmation
data.groupby("risk")[["Age", "Credit amount", "Duration", "Job"]].mean().round(2)


**What the output means:**
- **Duration separates best:** bad loans average ~25 months vs. ~19 for good ones — long loans go wrong more often.
- **Credit amount:** bad loans are larger on average (~3,938 vs. ~2,985 DM).
- **Age:** bad-risk applicants are slightly *younger* (~34 vs. ~36) — a mild signal.
- **Job** barely differs between outcomes — skill level alone predicts little here.


### 6.4 Categorical Features vs. the Target

**What we're about to do:** Compute the **bad-credit rate** within each category of `Sex`, `Housing`, `Checking account`, `Saving accounts`, and `Purpose`. Categories far above the overall 30% line are risk flags the model will likely learn.


In [ ]:
# Bad-credit rate per category, for five categorical features
cat_cols = ["Checking account", "Saving accounts", "Housing", "Purpose", "Sex"]
fig, axes = plt.subplots(1, 5, figsize=(20, 4.5))

for ax, col in zip(axes, cat_cols):
    rates = data.groupby(col)["risk"].mean().sort_values(ascending=False)
    bars = ax.barh(rates.index[::-1], rates.values[::-1] * 100, color=C_MAIN)
    for bar in bars:
        ax.text(bar.get_width() + 1.2, bar.get_y() + bar.get_height()/2,
                f"{bar.get_width():.0f}%", va="center", fontsize=9)
    ax.axvline(data["risk"].mean() * 100, color="gray", linestyle="--",
               linewidth=1)
    ax.set_title(col, fontsize=11)
    ax.set_xlabel("% bad credit")
    ax.set_xlim(0, 65)

fig.suptitle("Bad-credit rate by category (dashed line = overall 30%)", y=1.03)
sns.despine()
plt.tight_layout()
plt.show()


**What the output means (dashed line = overall 30% bad rate):**
- **Checking account is the star predictor:** applicants with `little` (overdrawn!) have a ~49% bad rate, `moderate` ~39%, `rich` ~22% — while `unknown` (no checking account) is safest of all at only ~12%. Counter-intuitive but real: in this data, having *no* checking account at the bank is safer than having any known balance.
- **Saving accounts:** `little` savings ~36% bad vs. `rich` ~12% — the safety-cushion story confirmed.
- **Housing:** renters and `free` housing are riskier than owners.
- **Purpose:** education and vacation/other loans are riskier; used-goods loans (radio/TV) safer.
- **Sex:** men ~28% vs. women ~35% bad rate — a **7-percentage-point gap** on our protected attribute. Bookmark this for Section 7.


### 6.5 Correlation Analysis

**What we're about to do:** Correlation heatmap of the numeric columns (diverging red–blue colormap, neutral at 0). With only 4 true numeric features this is a small matrix — the categorical features (analyzed above by group rates) are where much of this dataset's signal lives.


In [ ]:
# Correlation matrix of numeric features + target
numeric_cols = ["Age", "Job", "Credit amount", "Duration", "risk"]
corr = data[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(6.5, 5.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r",
            vmin=-1, vmax=1, center=0, square=True,
            cbar_kws={"label": "correlation"}, ax=ax)
ax.set_title("Correlation between numeric features and the target")
plt.tight_layout()
plt.show()


**What the output means (look at the `risk` row):**
- **`Duration` ≈ +0.21** — the strongest numeric predictor: longer loans → more bad outcomes.
- **`Credit amount` ≈ +0.15** — bigger loans → somewhat riskier.
- **`Age` ≈ −0.09** — older → slightly safer; weak.
- **`Job` ≈ 0.03** — essentially uninformative on its own.

Also note **`Credit amount` ↔ `Duration` ≈ +0.62**: big loans take longer to repay — our features are partly redundant. And **`Credit amount` ↔ `Job` ≈ +0.29**: higher-skilled applicants request larger loans.

Overall the numeric correlations are *weaker* than in COMPAS — in this dataset the categorical account-status features carry more of the signal.


### 6.6 Interesting Patterns & Observations

**What we're about to do:** Cross protected attribute × features: do men and women in this dataset differ on the features that predict risk? (Same proxy-feature check as COMPAS Section 7.6.)


In [ ]:
# Do the sexes differ on predictive features? (proxy check)
summary = (data.groupby("Sex")
             .agg(n=("Sex", "size"),
                  avg_age=("Age", "mean"),
                  avg_amount=("Credit amount", "mean"),
                  avg_duration=("Duration", "mean"),
                  bad_rate=("risk", "mean"))
             .round(2))
summary["bad_rate"] = (summary["bad_rate"] * 100).round(1)
summary


**What the output means:** Women in this dataset are on average **younger** (~33 vs. ~37) and request somewhat smaller loans, yet show a **higher bad-credit rate** (~35% vs. ~28%). Because age correlates with risk, `Age` acts as a **partial proxy** for `Sex` — dropping the `Sex` column would not fully remove sex-related disparity from a model's predictions. Same lesson as COMPAS: *fairness through unawareness doesn't work; you have to measure fairness explicitly.*

**One more caution:** remember `Sex` was **derived from a combined "personal status and sex" code** — e.g. all married men are one code, and the original data cannot distinguish single vs. married women in some categories. Real-world datasets often encode protected attributes messily like this.

**EDA takeaways:**
1. Small dataset (1,000 rows) → expect noisy statistics, especially per-group.
2. Clear class imbalance (70/30) → the "always say good" baseline is 70%; judge accuracy against that.
3. Best predictors: `Checking account` status, `Duration`, `Credit amount`, `Saving accounts`.
4. Informative missingness handled as an `"unknown"` category.
5. A 7-point bad-rate gap between the sexes exists in the raw data, with proxy features present.


---
## 7. Fairness-Related Analysis

### 7.1 Protected attributes in this dataset

| Attribute | Present as | Used in our project as |
|-----------|-----------|------------------------|
| **Sex** | `Sex` column | ⭐ the **sensitive feature** — all group-fairness metrics compare men vs. women |
| Age | `Age` column | a model feature (age is legally protected in credit decisions in many jurisdictions — e.g. the U.S. ECOA) |
| Foreign worker | in the raw UCI file (dropped in our 9 columns) | not used — but worth knowing it exists; nationality-linked attributes are protected too |

### 7.2 Why sex, and why does it matter here?

Credit is a legally regulated domain: laws like the U.S. **Equal Credit Opportunity Act** and EU anti-discrimination directives make it **illegal** to deny credit based on sex. Yet our EDA showed women in this dataset have a ~35% bad-credit rate vs. ~28% for men. A model trained naively on this data will learn that pattern — through the `Sex` column directly, or through proxies — and could systematically give women worse credit decisions.

Note the contrast with COMPAS: there the concern was **race** in criminal justice; here it is **sex** in finance. Same fairness machinery, different domain and attribute — that's the point of running the two studies in parallel.

### 7.3 How our project uses the protected attribute

In `German.py`, the `Sex` column of each split is set aside as `sensitive_features` and passed to the same metrics as COMPAS:

- **Group fairness** (`GroupFairness.py`): demographic parity difference, equalized odds difference, equal opportunity difference, disparate impact — each computed comparing **male vs. female**.
- **Individual fairness** (`IndividualFairness.py`): Theil index, generalized entropy, Atkinson index, Gini coefficient.
- Both enter the composite training loss: `total = α·BCE + (1−α)·(β·group + (1−β)·individual)`.

### 7.4 Known biases in this dataset

| Bias type | How it shows up here |
|-----------|----------------------|
| **Historical bias** | The labels record 1970s-era West German bank decisions and social structures (e.g. women's financial independence was legally restricted in Germany until 1962–1977!). The "ground truth" reflects that era. |
| **Representation bias** | 690 men vs. 310 women — statistics for women rest on far fewer examples, so they're noisier. |
| **Aggregation/encoding bias** | Sex arrives entangled with marital status in a single code. The "single female" code (A95) is defined in the documentation but **never occurs in the data** — every woman is lumped into one code (A92), while men get three separate marital-status codes. Women are literally recorded with less detail. |
| **Sampling bias** | These are 1,000 *approved* loans from one bank — people the bank rejected outright never appear. The model only ever learns about applicants the 1970s bank already deemed acceptable. |
| **Proxy bias** | Age (and loan size/purpose) correlate with sex, so removing the `Sex` column would not remove sex-linked patterns. |

**What we're about to do:** Compute the base rates per sex with group sizes — the raw material of every group-fairness metric — and visualize the gap.


In [ ]:
# Base rates by sex: the numbers every group-fairness metric is built on
base = (data.groupby("Sex")["risk"]
          .agg(n="size", bad_rate="mean"))
base["bad_rate_pct"] = (base["bad_rate"] * 100).round(1)
print(base[["n", "bad_rate_pct"]], "\n")

gap = (base.loc["female", "bad_rate"] - base.loc["male", "bad_rate"]) * 100
print(f"Base-rate gap (female − male): {gap:.1f} percentage points")

# Grouped bar chart: outcome distribution within each sex
ct = pd.crosstab(data["Sex"], data["risk"], normalize="index") * 100
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(2)
width = 0.38
b0 = ax.bar(x - width/2, ct[0], width, label="Good credit (0)", color=C_GOOD)
b1 = ax.bar(x + width/2, ct[1], width, label="Bad credit (1)", color=C_BAD)
for bars in (b0, b1):
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.2,
                f"{bar.get_height():.0f}%", ha="center", fontsize=10)
ax.set_xticks(x); ax.set_xticklabels(ct.index)
ax.set_ylabel("% of that sex's applicants")
ax.set_title("Loan outcome within each sex")
ax.set_ylim(0, 90)
ax.legend()
sns.despine()
plt.tight_layout()
plt.show()


**What the output means:** Within-group outcome rates: ~72% of men's loans were good vs. ~65% of women's — a base-rate gap of ~7 percentage points, computed from only 310 women (so with real statistical uncertainty).

**Why this matters for the experiments:** exactly as with COMPAS, unequal base rates mean an accurate model will naturally treat the groups differently (violating demographic parity), while forcing equal treatment costs accuracy somewhere. Our α/β sweep maps that trade-off. Because this dataset is small and the gap is modest, expect the fairness metrics here to be **noisier and the effects subtler** than in COMPAS.


---
## 8. Step-by-Step Workflow — From Raw Data to Model-Ready Tensors

The exact pipeline of `German.py`, step by step (deliberately identical in structure to `Compas.py`):

| Step | What | Why |
|------|------|-----|
| 1 | Load features + attach target from the source file | Cleaned CSV has no label; UCI file provides it |
| 2 | `fillna("unknown")` | Informative missingness becomes its own category |
| 3 | Select 9 features + `risk` | The study's feature set |
| 4 | Split 60/20/20 (train/val/test), `random_state=42` | Honest evaluation, reproducible |
| 5 | Set aside `Sex` per split | Needed to *measure* fairness on each split |
| 6 | Scale numeric + one-hot encode categorical (fit on train only) | Comparable numeric inputs, no leakage |
| 7 | Convert to PyTorch tensors | Format the training code expects |

Steps 1–3 are already done above. Let's do 4–7.

**What we're about to do (Step 4):** The double split: 80/20 first, then 75/25 of the 80% → 60/20/20 overall.


In [ ]:
from sklearn.model_selection import train_test_split

features = ["Age", "Sex", "Job", "Housing", "Saving accounts",
            "Checking account", "Credit amount", "Duration", "Purpose"]
target = "risk"

X = data[features]
y = data[target]

# 80% train+val vs 20% test, then 75/25 of the 80% -> 60/20/20 overall
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=42)

print(f"Train:      {X_train.shape[0]:>4} rows ({X_train.shape[0]/len(data)*100:.0f}%)")
print(f"Validation: {X_val.shape[0]:>4} rows ({X_val.shape[0]/len(data)*100:.0f}%)")
print(f"Test:       {X_test.shape[0]:>4} rows ({X_test.shape[0]/len(data)*100:.0f}%)")


**What the output means:** 600 / 200 / 200 rows. ⚠️ Note how small the test set is: with 200 people, **one** flipped prediction moves accuracy by 0.5 percentage points, and per-sex fairness metrics rest on roughly 140 men and 60 women. Keep that uncertainty in mind when reading the project's result heatmaps.


**What we're about to do (Step 5):** Keep the `Sex` column of each split in readable form for the fairness metrics.


In [ ]:
# Sensitive attribute per split (used by fairness metrics, kept readable)
sensitive_train = X_train["Sex"]
sensitive_val   = X_val["Sex"]
sensitive_test  = X_test["Sex"]

print("Sex distribution in the TEST split (fairness metrics computed on this):")
print(sensitive_test.value_counts())


**What the output means:** The test split's male/female ratio roughly mirrors the full data (≈ 2:1). The female group in the test set is small — another reason fairness numbers on this dataset wobble.


**What we're about to do (Step 6):** Preprocess: `StandardScaler` for numeric columns, `OneHotEncoder(drop='first')` for categorical ones. **Fit on train only**, transform all three splits (the no-leakage rule).


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Automatic type detection (same as German.py).
# Note: Job is int64, so it is treated as NUMERIC and scaled, not one-hot encoded.
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
print("Numeric:    ", numeric_features)
print("Categorical:", categorical_features)

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(drop="first"), categorical_features),
])

X_train_p = preprocessor.fit_transform(X_train)   # FIT on train only
X_val_p   = preprocessor.transform(X_val)
X_test_p  = preprocessor.transform(X_test)

print(f"\nColumns before preprocessing: {X_train.shape[1]}")
print(f"Columns after preprocessing:  {X_train_p.shape[1]}")


**What the output means:** 9 readable columns became **21 numeric ones**: 4 scaled numbers (`Age`, `Job`, `Credit amount`, `Duration`) plus one-hot columns for `Sex` (1), `Housing` (2), `Saving accounts` (4), `Checking account` (3), `Purpose` (7) — each category set minus its dropped reference. So `input_dim = 21` for the German model (vs. 13 for COMPAS).


**What we're about to do (Step 7):** Convert everything to PyTorch tensors — the final form the training loop consumes.


In [ ]:
import torch

X_train_t = torch.tensor(X_train_p, dtype=torch.float32)
X_val_t   = torch.tensor(X_val_p,   dtype=torch.float32)
X_test_t  = torch.tensor(X_test_p,  dtype=torch.float32)

y_train_t = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
y_val_t   = torch.tensor(y_val.values,   dtype=torch.float32).view(-1, 1)
y_test_t  = torch.tensor(y_test.values,  dtype=torch.float32).view(-1, 1)

print("Feature tensor:", X_train_t.shape, "| Label tensor:", y_train_t.shape)
print("\n✅ Data is now exactly in the form German.py feeds to the logistic-regression model.")


**What the output means:** Training tensors of shape `[600, 21]` and `[600, 1]`. From here, `German.py` runs the **identical experiment** to `Compas.py`: a logistic-regression model per (α, β, group-metric, individual-metric) combination — 196 models — writing results to `GERMAN_RESULTS/results.csv` and heatmaps to `GERMAN_RESULTS/PLOTS/`.


---
## 9. Summary — What We Learned

### The dataset in one paragraph
The German Credit dataset contains **1,000 loan applications** from a 1970s West German bank, each described by 9 personal/financial features, with a binary target **`risk`** (`1` = bad credit, 30% of cases). It's a classic ML benchmark and a standard fairness testbed because it includes legally protected attributes — our project uses **sex** as the sensitive feature.

### Key insights from our analysis

1. **The raw data needs decoding:** UCI's `A`-codes → readable categories; the target comes from the original file (good=1/bad=2 → our `risk` 0/1, with 1 = the adverse outcome, mirroring COMPAS).
2. **Missingness is informative:** ~39% have no known checking account; we encode missing as an `"unknown"` category rather than deleting or guessing values — and "unknown checking account" turns out to be *safer* than an overdrawn one!
3. **Class imbalance 70/30:** the always-say-good baseline scores 70%; judge model accuracy against that floor.
4. **Strongest predictors:** checking-account status, loan duration (r ≈ +0.21), credit amount (r ≈ +0.15), savings.
5. **Fairness gap in the raw data:** women's bad-credit rate ≈ 35% vs. men's ≈ 28%, with only 310 women in the data — a real but statistically noisy disparity, partly proxied by age.
6. **Small-data caveat everywhere:** 1,000 rows and a 200-row test set mean every metric — especially per-group fairness — carries visible statistical noise.
7. **Pipeline:** decode → fillna("unknown") → 9 features → 60/20/20 split → keep `Sex` aside → scale + one-hot (fit on train only) → 21-dim tensors.

### How this prepares us for the next stage

This notebook ends exactly where `German.py`'s modeling begins. The project trains logistic regression with the composite loss

`total = α·BCE + (1−α)·(β·group_fairness + (1−β)·individual_fairness)`

over the same **196-combination sweep** as COMPAS, producing accuracy/fairness heatmaps over α×β. Because the two datasets share one method, one model, and one encoding convention (1 = adverse outcome), you can now compare: *does the accuracy–fairness trade-off look the same for race in criminal justice (COMPAS, 7,214 rows) as for sex in credit scoring (German, 1,000 rows)?* — which is the heart of the FairML study.

📘 **Companion:** the **COMPAS Dataset notebook** follows this identical outline for the first dataset.
